# BONSAI HIOT: compare GHG footprints with MARIO

Calculate per-unit GHG footprints from BONSAI's native **b2 hybrid input-output table (HIOT)** and compare them with BONSAI's exported GWP100 results, matched by region and product. The last cell saves the comparison to CSV.


## 1. Prepare the environment

Install MARIO (the `mariopy` package) and Jupyter in a conda environment, then select that environment as the notebook kernel. See the [MARIO documentation](https://www.marioproject.com/en/latest/) for installation instructions.

The cell below checks for the optional UMFPACK solver and installs it from conda-forge if needed. If it installs packages, **restart the kernel and rerun the notebook from the beginning** before continuing.


In [ ]:
# Install umfpack if not already installed in your mario env
try:
    import scikits.umfpack
except ImportError:
    %conda install -y conda-forge::scikit-umfpack
    print("Restart the kernel and rerun everything before continuing.")

## 2. Locate the BONSAI files

1. Open [BONSAI Nextcloud](https://nx101805.your-storageshare.de/) and sign in with the access provided by the BONSAI team. This is the server's login page; contact the team if you do not yet have access to the shared `_bonsai` folder.
2. In `_bonsai`, open `_b Next version/HIOT/20260615`. Download and extract the release, or sync the folder to your computer.
3. Replace `/path/to/BONSAI` in the next cell with your local BONSAI folder. **`path` must point to the release directory containing the files**, not to the shared folder or a ZIP archive.

Only the following files are needed for this notebook; other files in the release can remain in the directory.

```text
BONSAI/
└── _b Next version/
    └── HIOT/
        └── 20260615/
            ├── extended_size_hiot.b2.pkl
            ├── final_demand.b2.csv
            ├── total_elementary_flow_w_iLUC.csv
            └── ghg_for_all.b2.GWP100.csv
```

| File | Used for |
| --- | --- |
| `extended_size_hiot.b2.pkl` | Native b2 technosphere read by the MARIO parser. |
| `final_demand.b2.csv` | Final demand read by the parser. |
| `total_elementary_flow_w_iLUC.csv` | Elementary flows, including iLUC, used to calculate GHG footprints. |
| `ghg_for_all.b2.GWP100.csv` | BONSAI's reference footprints, read directly in section 5. |

Use files from the same b2 release. `20260615` identifies the release folder, not the economic reference year. BONSAI's precomputed inverse files are not needed.


In [ ]:
from pathlib import Path
import pandas as pd

import mario

path = Path(
    "/path/to/BONSAI/_b Next version/HIOT/20260615"
)

## 3. Parse the native HIOT

`table="IOT"` imports the existing HIOT directly. MARIO preserves the native regional sector coverage, which can vary between regions. The next line explicitly selects UMFPACK as the solver.

The BONSAI parser defaults to `solve`, so footprint calculation does not require constructing the full Leontief inverse. If the parser reports unmatched rows, inspect `db.info["bonsai"]` for the import diagnostics.


In [ ]:
db = mario.parse_bonsai(path=path, table="IOT")
mario.set_linear_solver("umfpack")

## 4. Calculate the GHG footprint multipliers

`calc_ghg(label="GHG")` applies MARIO's BONSAI GHG profile, using **AR6 GWP100** by default, including BONSAI's characterized accelerated-CO2 flow. `keep_extensions(["GHG"])` retains only that aggregated satellite account; it does not reduce the size of the technosphere matrix.

`db.f` calculates the per-unit footprint multipliers. Dropping the common sector level leaves the `(region, product)` index used to match BONSAI's export. These are footprints per native product unit, rather than total final-demand footprints; units differ across products in a hybrid database.


In [ ]:
db.calc_ghg(label="GHG")
db.keep_extensions(["GHG"])
f_mario = db.f.loc["GHG", :].droplevel(1)
f_mario.index.names = ["region", "product"]

## 5. Read BONSAI's reference footprints

Read the **b2 GWP100** export from the same directory and reshape it to the same `(region, product)` index. This cell reads the precomputed footprint CSV; it does not load BONSAI's precomputed inverse.


In [ ]:
bonsai = pd.read_csv(path / "ghg_for_all.b2.GWP100.csv", index_col=[0, 1])
bonsai = bonsai.droplevel(1).stack(future_stack=True).dropna().swaplevel()
bonsai.index.names = ["region", "product"]

## 6. Compare and save the results

The output contains `MARIO`, `BONSAI` and `relative_difference`, calculated as `(MARIO - BONSAI) / BONSAI`. A value of `0.01` means MARIO is 1% higher. A zero BONSAI denominator produces `NaN`; a region/product present on only one side also remains visible with missing values.

The CSV is written as **`footprint_ghg_comparison.csv` in the notebook's current working directory**. Interpret differences after checking that the model release, elementary flows and characterization factors match; the comparison alone does not establish their cause.


In [ ]:
comparison = pd.concat(
    [f_mario.rename("MARIO"), bonsai.rename("BONSAI")], axis=1
).sort_index()
comparison["relative_difference"] = (comparison["MARIO"] - comparison["BONSAI"]) / comparison["BONSAI"].replace(0, float("nan"))
comparison.to_csv("footprint_ghg_comparison.csv")
comparison